# Notebook 05: Boolean Retrieval

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
**Boolean Retrieval** is the classic set-theoretic Information Retrieval model. Documents either match a boolean query (`True`) or do not (`False`). It provides exact, deterministic matching without scoring or ranking.

**Core Objectives:**
1. Load the 698 Phnom Penh restaurant inverted index.
2. Execute `AND` queries (intersection of postings lists).
3. Execute `OR` queries (union of postings lists).
4. Execute `NOT` queries (set difference against the universal document set).
5. Evaluate complex nested boolean queries with parentheses.
6. Observe benefits (exact precision) and limitations (no ranking/scoring).

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter
from src.preprocessing.text_preprocessing import TextPreprocessor
from src.indexing.inverted_index import InvertedIndex
from src.retrieval.boolean_retrieval import BooleanRetriever

print(f"Project root resolved: {PROJECT_ROOT}")

## 1. Load Dataset & Construct Inverted Index

In [ ]:
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

converter = DocumentConverter()
documents = converter.convert_all(records)
corpus = converter.get_corpus()

preprocessor = TextPreprocessor()
tokenized_corpus = preprocessor.preprocess_documents(corpus)

index = InvertedIndex()
index.build(tokenized_corpus)

retriever = BooleanRetriever(index, preprocessor)
print(f"Boolean Retriever ready with {index.num_documents} documents in the index.")

## 2. Test AND Retrieval

Find restaurants that contain **both** terms.

In [ ]:
query = "noodles AND soup"
matching_ids = retriever.search(query)
print(f"Query: '{query}' -> Found {len(matching_ids)} matching restaurants.")

for doc_id in matching_ids[:5]:
    rec = converter.get_record(doc_id)
    print(f" - [{doc_id}] {rec['name']} | Category: {rec['category']} | Address: {rec['address']}")

## 3. Test OR Retrieval

Find restaurants that contain **either** term (broader recall).

In [ ]:
query = "pizza OR burger"
matching_ids = retriever.search(query)
print(f"Query: '{query}' -> Found {len(matching_ids)} matching restaurants.")

for doc_id in matching_ids[:5]:
    rec = converter.get_record(doc_id)
    print(f" - [{doc_id}] {rec['name']} | Subcategories: {rec['sub_category']}")

## 4. Test NOT Retrieval (Negation)

Find Cambodian restaurants, but exclude those serving seafood.

In [ ]:
query = "cambodian AND NOT seafood"
matching_ids = retriever.search(query)
print(f"Query: '{query}' -> Found {len(matching_ids)} matching restaurants.")

for doc_id in matching_ids[:5]:
    rec = converter.get_record(doc_id)
    print(f" - [{doc_id}] {rec['name']} | Subcategories: {rec['sub_category']}")

## 5. Complex Nested Boolean Queries

Evaluate compound queries with grouping: `soup AND (noodles OR beef)`.

In [ ]:
complex_query = "soup AND (noodles OR beef)"
matching_ids = retriever.search(complex_query)
print(f"Query: '{complex_query}' -> Found {len(matching_ids)} matching restaurants.")

for doc_id in matching_ids[:5]:
    rec = converter.get_record(doc_id)
    print(f" - [{doc_id}] {rec['name']} | {rec['category']}")

## 6. Observations
- **Precision**: Boolean retrieval returns exact matches matching logical criteria.
- **Limitation**: All matching documents are returned unranked (there is no relevance score to tell us which matching restaurant is *best*).
- **Next Step**: In `06_tfidf.ipynb` and `07_bm25.ipynb`, we implement ranked retrieval using term frequency, document frequency, and document lengths!